In [0]:
# =============================================================================
# Silver · dim_municipio
# Fontes: bronze.frota_raw + bronze.rfb_municipios_raw + seed uf_regiao
# =============================================================================
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
 

In [0]:
%sql
USE CATALOG brazil_car_fleet;
USE SCHEMA silver;


In [0]:
# -----------------------------------------------------------------------------
# 1. Seed table: sigla_uf → nm_uf + nm_regiao (27 linhas, mantida no código)
# -----------------------------------------------------------------------------
 
uf_regiao_data = [
    ("AC", "ACRE",                "Norte"),
    ("AL", "ALAGOAS",             "Nordeste"),
    ("AM", "AMAZONAS",            "Norte"),
    ("AP", "AMAPÁ",               "Norte"),
    ("BA", "BAHIA",               "Nordeste"),
    ("CE", "CEARÁ",               "Nordeste"),
    ("DF", "DISTRITO FEDERAL",    "Centro-Oeste"),
    ("ES", "ESPÍRITO SANTO",      "Sudeste"),
    ("GO", "GOIÁS",               "Centro-Oeste"),
    ("MA", "MARANHÃO",            "Nordeste"),
    ("MG", "MINAS GERAIS",        "Sudeste"),
    ("MS", "MATO GROSSO DO SUL",  "Centro-Oeste"),
    ("MT", "MATO GROSSO",         "Centro-Oeste"),
    ("PA", "PARÁ",                "Norte"),
    ("PB", "PARAÍBA",             "Nordeste"),
    ("PE", "PERNAMBUCO",          "Nordeste"),
    ("PI", "PIAUÍ",               "Nordeste"),
    ("PR", "PARANÁ",              "Sul"),
    ("RJ", "RIO DE JANEIRO",      "Sudeste"),
    ("RN", "RIO GRANDE DO NORTE", "Nordeste"),
    ("RO", "RONDÔNIA",            "Norte"),
    ("RR", "RORAIMA",             "Norte"),
    ("RS", "RIO GRANDE DO SUL",   "Sul"),
    ("SC", "SANTA CATARINA",      "Sul"),
    ("SE", "SERGIPE",             "Nordeste"),
    ("SP", "SÃO PAULO",           "Sudeste"),
    ("TO", "TOCANTINS",           "Norte"),
]
 
df_uf_regiao = spark.createDataFrame(
    uf_regiao_data,
    schema=["sigla_uf", "nm_uf", "nm_regiao"]
)

In [0]:
# -----------------------------------------------------------------------------
# 2. UDF de normalização — remove acentos para equalizar os dois lados do join
#    O Spark não tem função nativa para isso, então usamos unicodedata
# -----------------------------------------------------------------------------
 
import unicodedata
 
@F.udf("string")
def remove_acentos(s):
    if s is None:
        return None
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

In [0]:
# -----------------------------------------------------------------------------
# 3. Ler e normalizar bronze.rfb_municipios_raw
#    Colunas esperadas no CSV: cd_municipio, nm_municipio, sigla_uf
# -----------------------------------------------------------------------------
 
df_rfb = spark.table("bronze.rfb_municipios")
 
df_rfb_norm = (
    df_rfb
    .select(
        F.col("cd_municipio").cast("string").alias("cd_ibge_municipio"),
        F.trim("nm_municipio_upper").alias("nm_municipio_upper"),
        F.upper(F.trim(F.col("sigla_uf"))).alias("sigla_uf"),
        F.col("nm_municipio_formatted")
    )
    .withColumn("nm_municipio_norm", remove_acentos(F.col("nm_municipio_upper")))
)

# cria coluna com o nome do estado sem acentos
df_uf_regiao = df_uf_regiao.withColumn("nm_uf_norm", remove_acentos(F.col("nm_uf")))

In [0]:
# -----------------------------------------------------------------------------
# 4. Extrair municípios distintos do frota_raw e normalizar
#    Colunas esperadas: nm_municipio, nm_uf
# -----------------------------------------------------------------------------
 
# Enriquecer com sigla_uf via seed (join por nm_uf)
df_frota_mun = (
    df_frota_mun
    .join(
        df_uf_regiao.select(
            F.col("sigla_uf"),
            F.col("nm_uf_norm").alias("nm_uf_norm_u")
        ),
        on=df_frota_mun["nm_uf_norm_f"] == F.col("nm_uf_norm_u"),
        how="left"
    )
    .drop("nm_uf_norm_f", "nm_uf_norm_u")
)

In [0]:
df_rfb_norm.printSchema()

In [0]:
df_frota_mun.printSchema()

In [0]:
# -----------------------------------------------------------------------------
# 5. Join principal: frota x RFB por nm_municipio_norm + sigla_uf
# -----------------------------------------------------------------------------

df_joined = (
    df_rfb_norm
    .join(
        df_uf_regiao.select("sigla_uf", "nm_uf", "nm_regiao", "nm_uf_norm"),
        on=[
            df_rfb_norm["sigla_uf"] == df_uf_regiao["sigla_uf"],
        ],
        how="left"
    )
    .select(
        df_rfb_norm["nm_municipio_norm"],
        df_uf_regiao["nm_uf"],
        df_rfb_norm["cd_ibge_municipio"],
        df_rfb_norm["nm_municipio_formatted"],
        df_uf_regiao["sigla_uf"],
        df_uf_regiao["nm_regiao"]
    )
).drop(df_rfb_norm["sigla_uf"])


In [0]:
df_joined.display()


In [0]:
# -----------------------------------------------------------------------------
# 6. Auditoria de match — inspecione antes de salvar
# -----------------------------------------------------------------------------
 
total          = df_joined.count()
sem_codigo     = df_joined.filter(F.col("cd_ibge_municipio").isNull()).count()
pct_sem_codigo = round(sem_codigo / total * 100, 2)
 
print(f"Total de municípios distintos na rfb : {total}")
print(f"Sem match na RFB (cd_ibge nulo)            : {sem_codigo} ({pct_sem_codigo}%)")
 
if sem_codigo > 0:
    print("\n--- Municípios sem match — verificar grafia ---")
    (
        df_joined
        .filter(F.col("cd_ibge_municipio").isNull())
        .select("*")
        .orderBy("nm_uf", "nm_municipio_norm")
        .show(100, truncate=False)
    )
# Fixed reference: select columns actually present after join, avoiding unresolved column error and ambiguity.

In [0]:
# -----------------------------------------------------------------------------
# 7. Gerar surrogate key e montar dim_municipio final
# -----------------------------------------------------------------------------

df_dim_municipio = (
    df_joined
    .withColumn(
        "id_municipio",
        F.row_number().over(Window.orderBy("sigla_uf", "nm_municipio_norm"))
    )
    .select(
        "id_municipio",       # surrogate key
        "cd_ibge_municipio",  # código RFB/IBGE — pode ser nulo se sem match
        "nm_municipio_norm",
        "nm_municipio_formatted",
        "nm_uf",
        "sigla_uf",
        "nm_regiao",
    )
)


In [0]:
# -----------------------------------------------------------------------------
# 8. Remover linha com código IBGE = '0' (EXTERIOR)
# -----------------------------------------------------------------------------


df_dim_municipio = df_dim_municipio.where(F.col("cd_ibge_municipio") != '0')

In [0]:
# -----------------------------------------------------------------------------
# 9. Salvar na Silver como Delta
# -----------------------------------------------------------------------------

(
    df_dim_municipio
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_municipio")
)
 
print("\nsilver.dim_municipio salva com sucesso.")
df_dim_municipio.show(10, truncate=False)